<a href="https://colab.research.google.com/github/Deepthi-Bhargavi-Kasturi/ai-learning-journey/blob/master/textgeneration_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf ~/.cache/huggingface/token
!rm -rf ~/.huggingface/token

from huggingface_hub import login
import math


#Install three essential python libraries created by HuggingFace for building, training and testing models
#transaformers: core library used to download, run and finetune state-of-the-art pre-trained AI models
#datasets: this library manages data needed to train/fine-tune or test the models
#evaluate: this helps evaluate the accuracy of the model's output - how well your model is performing.
!pip install transformers datasets evaluate

#LOAD ELI5 DATASET
#load_dataset is a function used to load a specific category of dataset
#It downloads and caches the data of the specified category.
from datasets import load_dataset

#dany0407/eli5_category:
#unique address to the dataset. <username> / <repository name>
#username is who uploaded or mirrorred this data set to HuggingFace and eli5_category is the repository : name of the dataset itself.
#This tells the data is from Reddit "Explain Like I'm Five"

#split="train[:5000]"
#We are loading first 5000 examples instead of entire dataset when starting with the fine-tuning
#train here indicates that we are accessing training data (not testing data) and want to partition training data
#train[:5000] - python slice method. Starting from index 0 to up to, but not including data at index 5000
#Examples: train[5000:10000] grabs the next 5000 rows from 5000 to 10000
#          train[-1000:] grabs only the last 1000 rows of the training set
#          validation[:10%] grabs the first 10% of the validation dataset.
# training data vs validation data set - after studying the training data, it tries to answer from the validation data set to measure how well the
# model is performing that is how well the model understood the concepts or if it just memorized the concepts
eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")


#the fetched first 5000 rows data set is divided into to train and test data sets
#train_test_split is a built in hugging face method - randomly shuffles the data and divides into 2 sections (train data, test data)
#test_size=0.2 -  test data is 20% and so train data will be 80% from 5000 rows (100%)
#the original single data set eli5 is now overridden by grouped dictionary structure containing both splits - train and test
#output looks like:
#   DatasetDict({
#     train: Dataset({
#          features: ['title', 'text', 'category'],
#          num_rows: 4000
#     }),
#     test: Dataset({
#           features: ['title', 'text', 'category'],
#           num_rows: 1000
#      })
#   })
#how to access data:
#   training_data = eli5["train"]
#   test_data = eli5["test"]
#   eli5["train"][0] - every single cell from the first row -- see the output it is entire one row.
eli5 = eli5.train_test_split(test_size=0.2)
print(eli5["train"][0])

#preprocessing - step 1
#flatten
eli5 = eli5.flatten()
print(eli5["train"][0])

#tokenize list of strings in the answers.text field using tokenizer() function.
#for this write a function. The parameter is the elements of eli5["train"] array that is at index 0, 1 etc
def preprocess_function(examples):
  return tokenizer([" ".join(x) for x in examples["answers.text"]])

#apply this function to entire dataset that is eli5["train"]
#for this use Datasets map method - loops over every single row / element in the dataset
#error at remove_columns = eli5.column_names: ValueError: Column to remove ['train', 'test'] not in the dataset. Current columns in the dataset: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers.a_id', 'answers.text', 'answers.score', 'answers.text_urls', 'title_urls', 'selftext_urls']
#batched = True -> this is to process multiple elements of datasets at once by grouping / batching them together. 1000 by default
#num_proc = 4 -> increase number of processes - tells the computer that we need 4 CPU porcessing cores to work on this task at the exact same time. It divides dataset into 4 equal pieces, works on them / runs them in parallel and stictches them back together
#remove_columns = eli5["train"].column_names -> err eli5.column_names -> eli5["train"].column_names: because we have split the data set into train and test #removes old text columns and keep only tokenized outputs
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = eli5["train"].column_names

)
#output at this step -> error -> [transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1109 > 1024). Running this sequence through the model will result in indexing errors
#Datasets comtains token sequences, some of these are longer than the specified maximum sequence length for this model (1109 > 1024).

#To solve this we use second preprocessing function
block_size = 128
def group_texts(examples):
  concatenated_examples = {
      k : sum(examples[k],[]) for k in examples.keys() #flattening of lists
  }

  total_length = len(concatenated_examples[list(examples.keys())[0]])
  if total_length >= block_size:
    total_length = (total_length // block_size)*block_size

  result = {
            k:[t[i:i+block_size] for i in range(0,total_length,block_size)]
            for k,t in concatenated_examples.items()
  }

  result["labels"] = result["input_ids"].copy()
  return result

#Apply above function over the entire dataset
lm_dataset = tokenized_eli5.map(group_texts, batched = True, num_proc = 4)

print(len(lm_dataset["train"][0]["input_ids"]))
#Output: 128

print(lm_dataset["train"][0]["input_ids"])
#Output: [2504, 338, 1444, 6588, 46314, 1358, 11, 290, 340, 318, 257, 1517, 356, 460, 466, 13, 1318, 561, 761, 284, 307, 281, 12964, 1581, 44, 20958, 2033, 286, 340, 284, 3753, 703, 881, 356, 821, 5137, 656, 262, 8137, 13, 632, 338, 4577, 284, 4646, 262, 6588, 5072, 13, 632, 561, 307, 588, 2111, 284, 12051, 5789, 748, 282, 1883, 6134, 284, 787, 4713, 1660, 780, 286, 674, 3236, 3512, 329, 10150, 75, 485, 14860, 13, 775, 460, 11, 475, 356, 6584, 470, 13, 383, 3037, 318, 44192, 1091, 276, 11, 475, 772, 611, 340, 547, 20823, 4166, 340, 561, 307, 2089, 284, 3494, 340, 13, 775, 10385, 11863, 290, 17173, 7718, 23461, 284, 7375, 17, 290, 1660, 284, 787, 2568, 13, 37941, 9482, 4898, 11, 33512, 11]

#DataCollation - padding batches
from transformers import DataCollatorForLanguageModeling
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

##Training
#Load the model
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()
eval_results = trainer.evaluate()

print(f"perplexity: {math.exp(eval_results['eval_loss']):.2f}")





   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 98.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.92MB            

...

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.914668,3.795102


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [3]:
from transformers import AutoTokenizer

# Load a model's tokenizer (e.g., SmolLM or GPT-2)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

text = "Fluffy felines effortlessly catch lasers."
encoded = tokenizer(text)

# Get the integer IDs
print("Token IDs:", encoded["input_ids"])

# Convert IDs back to individual text tokens
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))


Token IDs: [3968, 30977, 282, 11243, 69704, 2339, 72475, 13]
Tokens: ['Fl', 'uffy', 'Ġf', 'elines', 'Ġeffortlessly', 'Ġcatch', 'Ġlasers', '.']


In [3]:
#Inference
#Tokenize the text and return the input_ids as PyTorch tensors
#return_tensors (str or TensorType, optional) — If set, will return tensors instead of list of python integers. Acceptable values are: 'pt': Return PyTorch torch.Tensor objects. and 'np': Return Numpy np.ndarray objects.

from transformers import AutoTokenizer, AutoModelForCausalLM

prompt = "Somatic hypermutation allows the immune system to"

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")
inputs = tokenizer(prompt, return_tensors="pt").input_ids
print(inputs[0])
#Output: tensor([   50, 13795, 17508, 32071,  6276,   279, 22852,  1887,   311])


#outputs = tokenizer.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)
#Error: AttributeError: TokenizersBackend has no attribute generate
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM3-3B")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)
print(outputs)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

#Error: IndexError: tuple index out of range
#outputs = model.generate(inputs[0], max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

#Output
#tensor([[   50, 13795, 17508, 32071,  6276,   279, 22852,  1887,   311,  7068,
           #264, 17226, 77768,   315, 59854,    13,  1115,  1920,   374, 16996,
           #369,   279, 48232, 22852,  2077,    11, 28462,   279,  2547,   311,
         #15641,   323, 21277,   553,   264,  7029,  2134,   315, 78284,    13,
          #4452,    11,   279, 24717, 16940,  1794,   780, 17508, 32071,  7293,
         #31555, 16365,    13,  4815, 28528,   706,  6982,   430,   279,  1920,
         #18065,   279, 17219,   315,  4288, 34684,  1139,   279,  3977, 13918,
           #315, 33119, 76525, 24292, 21389,    11,   902,   527,  1243,  4183,
           #369,   872,  5845,   311, 10950,   311,  3230, 68937,   729,    13,
          # 578,  1401,  4311,   304,   420,  1920,   527,   279,   362,   926,
         #49242,    11,   902, 40019,   279, 34684,    11,   323,   279]])
#['Somatic hypermutation allows the immune system to generate a diverse repertoire of antibodies. This process is crucial for the adaptive immune response, enabling the body to recognize and neutralize a wide range of pathogens. However, the mechanisms underlying somatic hypermut#





tensor([   50, 13795, 17508, 32071,  6276,   279, 22852,  1887,   311])


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


tensor([[   50, 13795, 17508, 32071,  6276,   279, 22852,  1887,   311,  7068,
           264, 17226, 77768,   315, 59854,    13,  1115,  1920,   374, 16996,
           369,   279, 48232, 22852,  2077,    11, 28462,   279,  2547,   311,
         15641,   323, 21277,   553,   264,  7029,  2134,   315, 78284,    13,
          4452,    11,   279, 24717, 16940,  1794,   780, 17508, 32071,  7293,
         31555, 16365,    13,  4815, 28528,   706,  6982,   430,   279,  1920,
         18065,   279, 17219,   315,  4288, 34684,  1139,   279,  3977, 13918,
           315, 33119, 76525, 24292, 21389,    11,   902,   527,  1243,  4183,
           369,   872,  5845,   311, 10950,   311,  3230, 68937,   729,    13,
           578,  1401,  4311,   304,   420,  1920,   527,   279,   362,   926,
         49242,    11,   902, 40019,   279, 34684,    11,   323,   279]])
['Somatic hypermutation allows the immune system to generate a diverse repertoire of antibodies. This process is crucial for the adaptiv